# 🔎 VeriFile AI — Hacktoberfest Medan 2026

**Mode 1 (tanpa GPU):** analisis metadata / PDF / DOCX / JPG, PDF cover-and-replace checks, RapidOCR amount regions, OpenCV local texture / photo-of-phone-screen cues / before-after comparison + JSON report. **Mode 2 (opsional GPU):** Gemma 4 E2B multimodal untuk membaca dan mendeskripsikan dugaan anomali. Tidak bisa memastikan transaksi asli, penipuan, atau gambar AI.

**Cara memakai:** unduh `VeriFile-AI-starter.zip` dari ChatGPT → buka notebook di [Google Colab](https://colab.research.google.com/) → *Runtime → Change runtime type → T4 GPU* (opsional) → jalankan cell berurutan. Mode tanpa GPU tetap berjalan.

⚠️ Untuk keamanan, hanya pakai dokumen sintetis/berizin. Public Gradio link tidak cocok untuk data keuangan pribadi.

In [ ]:
import os, sys, shutil, subprocess
print('Python:', sys.version.split()[0])
try:
    import torch
    print('GPU aktif:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('Device:', torch.cuda.get_device_name(0))
except ImportError:
    print('Torch belum diinstal; mode forensik tetap bisa digunakan.')

## 1. Unggah starter ZIP (satu kali per runtime)

Pilih file `VeriFile-AI-starter.zip`. Notebook dan ZIP diunduh pada komputer Anda; jalankan notebook ini setelah membuka Colab. Tidak memerlukan URL GitHub yang belum dibuat.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()  # pilih VeriFile-AI-starter.zip
zip_candidates = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_candidates:
    raise ValueError('Unggah ZIP starter project. Bukan file PDF atau gambar.')
zip_name = zip_candidates[0]
with zipfile.ZipFile(zip_name) as z:
    members = z.infolist()
    if len(members) > 500:
        raise ValueError('Terlalu banyak file di dalam ZIP.')
    for m in members:
        from pathlib import PurePosixPath
        normalized = PurePosixPath(m.filename)
        if m.filename.startswith('/') or '..' in normalized.parts:
            raise ValueError('Path arsip tidak aman.')
    z.extractall('/content')
os.chdir('/content/VeriFile-AI')
print('Project directory:', os.getcwd())

## 2. Instal dependensi, OCR dan uji forensic engine

Colab memiliki akses internet untuk mengunduh RapidOCR/ONNX Runtime. Integrasi OCR di bundel ini belum teruji karena jaringan lingkungan pembuat tidak tersedia. Jika gagal, gunakan `requirements.txt` dan mode tanpa OCR, atau lakukan perbaikan berdasarkan traceback.


In [ ]:
%pip -q install -r requirements-ocr.txt
!python demo/create_samples.py
!python -m pytest -q
!python -m verifile.cli demo/edited_overlay_sample.pdf --output demo/report_demo.json


## 3. Opsional: aktifkan Gemma 4 multimodal (GPU) dan/atau fallback Gemini API (tanpa GPU)

- **Utama (GPU):** Model default `google/gemma-4-E2B-it` membutuhkan *T4 GPU* dan sekitar 10 GB storage.
- **Fallback otomatis (CPU/GPU):** Simpan `GEMINI_API_KEY` di Colab **Secrets** (ikon kunci di kiri). Bila GPU tidak tersedia atau Gemma lokal gagal di-load, aplikasi otomatis jatuh (fallback) ke **Gemini API** (`gemini-2.5-flash`) tanpa perlu GPU atau download model besar.

In [ ]:
# 1) Konfigurasi fallback Gemini API dari Colab Secrets (berjalan di CPU maupun GPU).
has_gemini_fallback = False
try:
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY')
    if gemini_key:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-gemini.txt'], check=True)
        os.environ['GEMINI_API_KEY'] = gemini_key
        os.environ['GEMINI_FALLBACK'] = '1'
        os.environ.setdefault('GEMINI_MODEL_ID', 'gemini-2.5-flash')
        del gemini_key
        has_gemini_fallback = True
        print('Fallback Gemini API aktif (Colab Secret GEMINI_API_KEY ditemukan).')
except Exception as e:
    print('GEMINI_API_KEY belum diset di Colab Secrets (opsional untuk fallback tanpa GPU):', type(e).__name__)

# 2) SET True hanya jika ingin mencoba Gemma 4 lokal di Colab GPU sebagai model utama.
ENABLE_GEMMA = False
if ENABLE_GEMMA:
    import torch
    if not torch.cuda.is_available():
        if has_gemini_fallback:
            print('GPU tidak aktif; saat checkbox AI dicentang di UI, app akan otomatis memakai fallback Gemini API.')
        else:
            raise RuntimeError('Pilih T4 GPU di Runtime > Change runtime type, atau set Colab Secret GEMINI_API_KEY untuk fallback.')
    else:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-gemma.txt'], check=True)
        os.environ['GEMMA_4BIT'] = '1'
        os.environ['GEMMA_MODEL_ID'] = 'google/gemma-4-E2B-it'
        try:
            from google.colab import userdata
            hf_token = userdata.get('HF_TOKEN')  # Colab > Secrets; jangan disimpan di notebook.
            if hf_token:
                from huggingface_hub import login
                login(token=hf_token, add_to_git_credential=False)
                del hf_token
                print('Hugging Face authentication configured.')
        except Exception as e:
            print('Colab secret HF_TOKEN belum diset (jika model membutuhkan akses, set token):', type(e).__name__)
        print('Gemma siap di-load ketika checkbox UI ON (dengan fallback Gemini API jika diset).')
else:
    if has_gemini_fallback:
        print('Gemma lokal dilewati; checkbox AI di UI siap memakai fallback Gemini API tanpa GPU.')
    else:
        print('Skip Gemma & Gemini. Forensic engine tetap berjalan tanpa GPU.')

## 4. Jalankan demo Gradio

Untuk melihat app dari browser di Colab, link share dapat dipakai; **jangan upload dokumen finansial nyata** pada demo publik. Bila share tidak dipilih, jalankan dengan fasilitas proxy Colab (jika tersedia). Setelah selesai, tutup tab dan matikan runtime.

In [ ]:
# Boleh true HANYA untuk contoh sintetis; link publik berpotensi diakses pihak lain.
SHARE_PUBLIC_DEMO = False
import app as verifile_app
verifile_app.build_app().launch(share=SHARE_PUBLIC_DEMO, debug=True, max_file_size='20mb')

## Checklist demo hack day

1. Run tests (`21 passed` in the authoring environment; re-test Colab).
2. Compare `demo/edited_sample.jpg` with `demo/clean_sample.jpg` as the **same-transaction original**; inspect highlighted differences.
3. Upload `demo/edited_overlay_sample.pdf` and inspect the `pdf_text_cover_replace` finding: the file has retained text → patch → replacement text paint order.
4. Use RapidOCR highlights and the local-texture heatmap as *review cues*, **not as proof** of fraud or AI generation. The first OCR invocation may initialize models.
5. If GPU and sufficient disk space exist, enable Gemma 4 and review full screenshot + crops of probable amount areas.
6. Explain that only an official bank ledger/authorized API can verify transfer settlement.

Event: https://events.mlh.com/events/15226  |  Gemma docs: https://ai.google.dev/gemma/docs/core/huggingface_inference

7. Upload `demo/screen_photo_illustration.jpg`: inspect `forensics.recapture_diagnostics` and the candidate outline. It is synthetic artwork, not a photograph, and does not validate real-world recapture detection. Real camera date refers to capture time, not transaction time.
